In [ ]:
#
# For licensing see accompanying LICENSE file.
# Copyright (C) 2026 Apple Inc. All Rights Reserved.
#

In [ ]:
import os
from pathlib import Path

NB_DIR = Path(os.getcwd())
ROOT_DIR = NB_DIR.parent 

In [2]:
cd $ROOT_DIR

/Users/mohammedsaeed/Desktop/Projects/lang_bench


In [3]:
import json
import pandas as pd
from pathlib import Path

from scipy.stats import hmean

from src.utils.helpers import  get_result_dict, highlight_accuracy_pos_neg, highlight_gradient

In [4]:
DATA_GEN_PATH=Path("data/generated")
RESULT_DIR = Path("data/results")


MODELS =  ["gemini-2.0-flash-lite", 
           "gemini-2.0-flash", 
           "CohereLabs/aya-expanse-32b",
           "microsoft/Phi-4-mini-instruct", 
           "utter-project/EuroLLM-9B-Instruct", 
           "google/gemma-3-27b-it",
           "Qwen/Qwen2.5-32B-Instruct",
           "Qwen/Qwen3-32B"]

In [5]:
# num evaluations
num_tem_scenario = len(list(DATA_GEN_PATH.glob("*/*/*")))
len(MODELS)* num_tem_scenario * 2 + num_tem_scenario

850

In [6]:
# num test instances
sum([len(json.load(open(x,'r'))['sampled_data']) for x in list(DATA_GEN_PATH.rglob("**/sampled_data.json"))])

32789

# Check No Missing Files

In [7]:
# CHECK ALL FILES ARE THERE WITH NO FAILS FOR NON-THIKNING MODELS
generated_sampled_files = DATA_GEN_PATH.rglob("**/sampled_data.json")
for generated_sample_file in generated_sampled_files:
    for model in MODELS:
        for prompt_method in ["direct","cot"]:
            result_sample_file = Path(str(generated_sample_file.parent).replace("generated","results")+'/'+model)
            template_res_file = list(result_sample_file.rglob(f"{prompt_method}**/**/results.json"))
            assert len(template_res_file)==1,f"Multiple result files for {template_res_file}"
            is_success = [x['prediction']['success'] for x in json.load(open(template_res_file[0],'r'))['results']]
            if not all(is_success): print(f"Error in {template_res_file[0]}")


In [8]:
# CHECK FOR THINK MODEL
generated_sampled_files = DATA_GEN_PATH.rglob("**/sampled_data.json")
for generated_sample_file in generated_sampled_files:
    for model in ["Qwen/Qwen3-32B"]:
        for prompt_method in ["think"]:
            result_sample_file = Path(str(generated_sample_file.parent).replace("generated","results")+'/'+model)
            template_res_file = list(result_sample_file.rglob(f"{prompt_method}**/**/results.json"))
            assert len(template_res_file)==1,f"Multiple result files for {template_res_file}"
            if len(template_res_file)!=1:
                print(f"The number of directories is {len(template_res_file)} when it should be one. Check {result_sample_file} with {model}/{prompt_method}")
                print(f"Run the following command:\npython scripts/run_inference.py  --file {generated_sample_file} --scenario {'generation' if 'generation' in str(generated_sample_file) else 'judge'} system_method={prompt_method} model={model}")
                continue
            inference_ts_dir = template_res_file[0]
            is_success = [x['prediction']['success'] for x in json.load(open(template_res_file[0],'r'))['results']]
            if not all(is_success): print(f"Error in {template_res_file[0]}")

# Aggregated Table

In [9]:
# Helper Functions
def convert_template_name(text: str):
    if "indicative" in text.lower(): return "com-1"
    if "imperative" in text.lower(): return "com-2"
    if "adjective" in text.lower(): return "com-3"
    return text


def get_template_summary_df(prompt_method_result_dict):
    records = []

    for model_name, prompts in prompt_method_result_dict.items():
        for prompt_type, langs in prompts.items():
            for lang, categories in langs.items():
                for category, results in categories.items():
                    gen = hmean([x['accuracy'] for x in results['generation']])
                    jy = hmean([x['accuracy'] for x in results['judge'] if x['bucket_key'].endswith('_Yes')])
                    jn = hmean([x['accuracy'] for x in results['judge'] if x['bucket_key'].endswith('_No')])
                    for measure, val in zip(['Gen', 'JY', 'JN'], [gen, jy, jn]):
                        records.append({
                            'category': lang+'-'+convert_template_name(category),
                            'model': model_name+"_"+prompt_type,
                            'measure': measure,
                            'value': val
                        })

    df = pd.DataFrame(records)
    summary_df = df.pivot(index='category', columns=['model', 'measure'], values='value').round(3)
    lang_order = {'eng': 0, 'ara': 1, 'heb': 2, 'rus': 3, 'tur': 4, 'fin': 5}
    summary_df_sorted = summary_df.sort_index(key=lambda idx: idx.map(lambda x: lang_order[x.split('-')[0]]))
    return summary_df_sorted

## Direct vs CoT

In [10]:
# get results
result_dicts = {}
for prompting_method in ["direct", "cot"]:
    pm_result_dict = get_result_dict(MODELS,prompting_method, RESULT_DIR, output_files=True)
    result_dicts[prompting_method] = pm_result_dict

### Absolute DFs

In [11]:
# get aggregated dfs
prompting_method_dfs = {}
for prompting_method in ["direct", "cot"]:
    prompting_method_dfs[prompting_method] = []
    summary_df = get_template_summary_df(result_dicts[prompting_method])
    prompting_method_dfs[prompting_method].append(summary_df.copy())
    display(summary_df.style.map(highlight_gradient))

    # average over language
    summary_df['Prefix']=summary_df.index.str.split('-',n=1).str[0]
    lang_agg_summary_df = summary_df.groupby('Prefix').mean()
    prompting_method_dfs[prompting_method].append(lang_agg_summary_df.copy())
    display(lang_agg_summary_df.style.map(highlight_gradient))

    # Calculate Judgement Power
    df_models = lang_agg_summary_df.columns.get_level_values(0).unique()
    gen_judge_power_df = lang_agg_summary_df.copy()
    for df_model in df_models:
        gen_judge_power_df[(df_model,'Jud')] = gen_judge_power_df[[(df_model,  'JY'),(df_model,  'JN')]].apply(lambda x: hmean(x),axis=1)

    ordered_cols = [(df_model, col) for df_model in df_models for col in ['Gen', 'JY', 'JN', 'Jud']]
    gen_judge_power_df = gen_judge_power_df[ordered_cols]

    # Calculate Model Power
    df_models = gen_judge_power_df.columns.get_level_values(0).unique()
    model_power_df = gen_judge_power_df.copy()
    for df_model in df_models:
        model_power_df[(df_model,'MP')] = model_power_df[[(df_model,  'Gen'),(df_model,  'Jud')]].apply(lambda x: hmean(x),axis=1)
    ordered_cols = [(df_model, col) for df_model in df_models for col in ['Gen', 'JY','JN', 'Jud','MP']]
    model_power_df = model_power_df[ordered_cols]
    prompting_method_dfs[prompting_method].append(model_power_df.copy())
    display(model_power_df.style.map(highlight_gradient))


    # Calculate Model Power Across langauges
    df_models = model_power_df.columns.get_level_values(0).unique()
    mean_model_power_df = model_power_df.copy()

    # We Remove English
    mean_model_power_df = mean_model_power_df[mean_model_power_df.index!='eng']
    mean_model_power_df.loc['hmean'] = mean_model_power_df.apply(lambda x: hmean(x),axis=0)
    mean_model_power_df.loc['mean'] = mean_model_power_df.mean()

    ordered_cols = [(df_model, col) for df_model in df_models for col in ['Gen','JY','JN','Jud','MP']]
    mean_model_power_df = mean_model_power_df[ordered_cols]
    mean_model_power_df = mean_model_power_df.loc[['hmean','mean']]
    prompting_method_dfs[prompting_method].append(mean_model_power_df.copy())
    display(mean_model_power_df.style.map(highlight_gradient))

    print()
    print()
    print()


/var/folders/b1/j3sqn5ms737fg97dhk9w3b5m0000gn/T/ipykernel_67492/3206989047.py:18: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  jn = hmean([x['accuracy'] for x in results['judge'] if x['bucket_key'].endswith('_No')])


/var/folders/b1/j3sqn5ms737fg97dhk9w3b5m0000gn/T/ipykernel_67492/3775911564.py:11: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  lang_agg_summary_df = summary_df.groupby('Prefix').mean()


/var/folders/b1/j3sqn5ms737fg97dhk9w3b5m0000gn/T/ipykernel_67492/3206989047.py:18: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  jn = hmean([x['accuracy'] for x in results['judge'] if x['bucket_key'].endswith('_No')])


/var/folders/b1/j3sqn5ms737fg97dhk9w3b5m0000gn/T/ipykernel_67492/3775911564.py:11: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  lang_agg_summary_df = summary_df.groupby('Prefix').mean()


### Max/Min DFs

In [12]:
# Direct vs CoT Max/Min
assert len(prompting_method_dfs['direct']) == len(prompting_method_dfs['cot'])

df1 = prompting_method_dfs['cot'][0]
df2 = prompting_method_dfs['direct'][0]
assert df1.index.equals(df2.index)

columns = pd.MultiIndex.from_tuples([
    (col[0].removesuffix('_cot'), col[1]) for col in df1.columns
])
df_diff = pd.DataFrame(df1.values - df2.values, index=df1.index, columns=columns)
display(df_diff.style.map(highlight_accuracy_pos_neg))

# average over language
df_diff['Prefix']=df_diff.index.str.split('-',n=1).str[0]
max_lang_agg_summary_df = df_diff.groupby('Prefix').max()
display(max_lang_agg_summary_df.style.map(highlight_accuracy_pos_neg))
print(max_lang_agg_summary_df.to_latex())

min_lang_agg_summary_df = df_diff.groupby('Prefix').min()
display(min_lang_agg_summary_df.style.map(highlight_accuracy_pos_neg))
print(min_lang_agg_summary_df.to_latex())


/var/folders/b1/j3sqn5ms737fg97dhk9w3b5m0000gn/T/ipykernel_67492/1526048751.py:16: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  max_lang_agg_summary_df = df_diff.groupby('Prefix').max()


\begin{tabular}{lrrrrrrrrrrrrrrrrrrrrrrrr}
\toprule
 & \multicolumn{3}{r}{gemini-2.0-flash-lite} & \multicolumn{3}{r}{gemini-2.0-flash} & \multicolumn{3}{r}{CohereLabs/aya-expanse-32b} & \multicolumn{3}{r}{microsoft/Phi-4-mini-instruct} & \multicolumn{3}{r}{utter-project/EuroLLM-9B-Instruct} & \multicolumn{3}{r}{google/gemma-3-27b-it} & \multicolumn{3}{r}{Qwen/Qwen2.5-32B-Instruct} & \multicolumn{3}{r}{Qwen/Qwen3-32B} \\
 & Gen & JY & JN & Gen & JY & JN & Gen & JY & JN & Gen & JY & JN & Gen & JY & JN & Gen & JY & JN & Gen & JY & JN & Gen & JY & JN \\
Prefix &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
ara & 0.677000 & 0.033000 & 0.395000 & 0.893000 & 0.270000 & 0.581000 & 0.397000 & 0.305000 & 0.365000 & 0.000000 & 0.389000 & 0.136000 & 0.175000 & 0.025000 & 0.136000 & 0.408000 & -0.018000 & 0.470000 & 0.602000 & 0.385000 & 0.182000 & 0.625000 & 0.338000 & 0.338000 \\
eng & 0.000000 & 0.485000 & 0.000000 & 0.010000 & 0.041000 & 0.000000 & 0.142000

/var/folders/b1/j3sqn5ms737fg97dhk9w3b5m0000gn/T/ipykernel_67492/1526048751.py:20: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  min_lang_agg_summary_df = df_diff.groupby('Prefix').min()


\begin{tabular}{lrrrrrrrrrrrrrrrrrrrrrrrr}
\toprule
 & \multicolumn{3}{r}{gemini-2.0-flash-lite} & \multicolumn{3}{r}{gemini-2.0-flash} & \multicolumn{3}{r}{CohereLabs/aya-expanse-32b} & \multicolumn{3}{r}{microsoft/Phi-4-mini-instruct} & \multicolumn{3}{r}{utter-project/EuroLLM-9B-Instruct} & \multicolumn{3}{r}{google/gemma-3-27b-it} & \multicolumn{3}{r}{Qwen/Qwen2.5-32B-Instruct} & \multicolumn{3}{r}{Qwen/Qwen3-32B} \\
 & Gen & JY & JN & Gen & JY & JN & Gen & JY & JN & Gen & JY & JN & Gen & JY & JN & Gen & JY & JN & Gen & JY & JN & Gen & JY & JN \\
Prefix &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
ara & -0.702000 & -0.123000 & 0.000000 & -0.449000 & 0.063000 & -0.254000 & -0.189000 & 0.025000 & -0.107000 & -0.208000 & -0.105000 & -0.384000 & -0.037000 & -0.044000 & 0.000000 & -0.683000 & -0.406000 & 0.273000 & -0.382000 & -0.220000 & -0.158000 & -0.532000 & -0.117000 & -0.035000 \\
eng & 0.000000 & 0.000000 & 0.000000 & -0.015000 & 0.000000 & 

### Difference DFs

In [13]:
#  CoT/Direct Difference
assert len(prompting_method_dfs['direct']) == len(prompting_method_dfs['cot'])
num = len(prompting_method_dfs['direct'])
for i in range(num):
    df1 = prompting_method_dfs['cot'][i]
    df2 = prompting_method_dfs['direct'][i]
    assert df1.index.equals(df2.index)

    columns = pd.MultiIndex.from_tuples([
        (col[0].removesuffix('_cot'), col[1]) for col in df1.columns
    ])
    df_diff = pd.DataFrame(df1.values - df2.values, index=df1.index, columns=columns)
    display(df_diff.style.map(highlight_accuracy_pos_neg))
    print(df_diff.round(3).to_latex())


\begin{tabular}{lrrrrrrrrrrrrrrrrrrrrrrrr}
\toprule
 & \multicolumn{3}{r}{gemini-2.0-flash-lite} & \multicolumn{3}{r}{gemini-2.0-flash} & \multicolumn{3}{r}{CohereLabs/aya-expanse-32b} & \multicolumn{3}{r}{microsoft/Phi-4-mini-instruct} & \multicolumn{3}{r}{utter-project/EuroLLM-9B-Instruct} & \multicolumn{3}{r}{google/gemma-3-27b-it} & \multicolumn{3}{r}{Qwen/Qwen2.5-32B-Instruct} & \multicolumn{3}{r}{Qwen/Qwen3-32B} \\
 & Gen & JY & JN & Gen & JY & JN & Gen & JY & JN & Gen & JY & JN & Gen & JY & JN & Gen & JY & JN & Gen & JY & JN & Gen & JY & JN \\
category &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
eng-com-1 & 0.000000 & 0.000000 & 0.000000 & -0.015000 & 0.000000 & 0.000000 & -0.106000 & -0.033000 & -0.084000 & 0.015000 & 0.437000 & -0.087000 & 0.036000 & -0.021000 & -0.461000 & -0.005000 & 0.000000 & 0.010000 & 0.000000 & 0.084000 & 0.000000 & 0.020000 & 0.016000 & 0.000000 \\
eng-com-2 & 0.000000 & 0.120000 & NaN & 0.000000 & 0.020000 & NaN

\begin{tabular}{lrrrrrrrrrrrrrrrrrrrrrrrr}
\toprule
 & \multicolumn{3}{r}{gemini-2.0-flash-lite} & \multicolumn{3}{r}{gemini-2.0-flash} & \multicolumn{3}{r}{CohereLabs/aya-expanse-32b} & \multicolumn{3}{r}{microsoft/Phi-4-mini-instruct} & \multicolumn{3}{r}{utter-project/EuroLLM-9B-Instruct} & \multicolumn{3}{r}{google/gemma-3-27b-it} & \multicolumn{3}{r}{Qwen/Qwen2.5-32B-Instruct} & \multicolumn{3}{r}{Qwen/Qwen3-32B} \\
 & Gen & JY & JN & Gen & JY & JN & Gen & JY & JN & Gen & JY & JN & Gen & JY & JN & Gen & JY & JN & Gen & JY & JN & Gen & JY & JN \\
Prefix &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
ara & -0.066000 & -0.023000 & 0.238000 & 0.116000 & 0.136000 & 0.122000 & 0.143000 & 0.164000 & 0.044000 & -0.069000 & 0.115000 & -0.095000 & 0.078000 & 0.002000 & 0.049000 & -0.167000 & -0.200000 & 0.365000 & 0.061000 & 0.118000 & -0.008000 & 0.077000 & 0.109000 & 0.124000 \\
eng & 0.000000 & 0.202000 & 0.000000 & -0.002000 & 0.020000 & 0.000000 & -

\begin{tabular}{lrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrr}
\toprule
 & \multicolumn{5}{r}{gemini-2.0-flash-lite} & \multicolumn{5}{r}{gemini-2.0-flash} & \multicolumn{5}{r}{CohereLabs/aya-expanse-32b} & \multicolumn{5}{r}{microsoft/Phi-4-mini-instruct} & \multicolumn{5}{r}{utter-project/EuroLLM-9B-Instruct} & \multicolumn{5}{r}{google/gemma-3-27b-it} & \multicolumn{5}{r}{Qwen/Qwen2.5-32B-Instruct} & \multicolumn{5}{r}{Qwen/Qwen3-32B} \\
 & Gen & JY & JN & Jud & MP & Gen & JY & JN & Jud & MP & Gen & JY & JN & Jud & MP & Gen & JY & JN & Jud & MP & Gen & JY & JN & Jud & MP & Gen & JY & JN & Jud & MP & Gen & JY & JN & Jud & MP & Gen & JY & JN & Jud & MP \\
Prefix &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
ara & -0.066000 & -0.023000 & 0.238000 & 0.267000 & 0.204000 & 0.116000 & 0.136000 & 0.122000 & 0.128000 & 0.122000 & 0.143000 & 0.164000 & 0.044000 & 0.100000 & 0.140000 & -0.069000 & 0.115000 & -0.0

\begin{tabular}{lrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrr}
\toprule
 & \multicolumn{5}{r}{gemini-2.0-flash-lite} & \multicolumn{5}{r}{gemini-2.0-flash} & \multicolumn{5}{r}{CohereLabs/aya-expanse-32b} & \multicolumn{5}{r}{microsoft/Phi-4-mini-instruct} & \multicolumn{5}{r}{utter-project/EuroLLM-9B-Instruct} & \multicolumn{5}{r}{google/gemma-3-27b-it} & \multicolumn{5}{r}{Qwen/Qwen2.5-32B-Instruct} & \multicolumn{5}{r}{Qwen/Qwen3-32B} \\
 & Gen & JY & JN & Jud & MP & Gen & JY & JN & Jud & MP & Gen & JY & JN & Jud & MP & Gen & JY & JN & Jud & MP & Gen & JY & JN & Jud & MP & Gen & JY & JN & Jud & MP & Gen & JY & JN & Jud & MP & Gen & JY & JN & Jud & MP \\
Prefix &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
hmean & -0.128000 & 0.059000 & 0.150000 & 0.146000 & 0.042000 & 0.040000 & 0.075000 & 0.076000 & 0.078000 & 0.062000 & -0.045000 & 0.067000 & 0.208000 & 0.166000 & 0.029000 & -0.021000 & 0.080000 & -0

### Validity Rate

In [14]:
validity_df = []
for scenario in ["generation","judge"]:
    for prompting_method in ["direct","cot"]:
        print(scenario, prompting_method)
        print("="*20)
        for model in MODELS:
            num, den = 0, 0
            for summary_file in RESULT_DIR.glob(f"**/{scenario}/**/{model}/{prompting_method}**/**/summary.csv"):
                df = pd.read_csv(summary_file)
                all_row = df[df.bucket_key=="ALL"].squeeze()
                num+=all_row['n_valid']
                den+=all_row['n_total']
            print(model)
            print(f'Validity Rate: {num/den}')
            print(f'Invalid Number: {den-num}/{den}')
            if scenario == "judge" and prompting_method=="cot":
                validity_df.append([model,num/den])
            print()

generation direct
gemini-2.0-flash-lite
Validity Rate: 1.0
Invalid Number: 0/6608

gemini-2.0-flash
Validity Rate: 1.0
Invalid Number: 0/6608

CohereLabs/aya-expanse-32b
Validity Rate: 1.0
Invalid Number: 0/6608

microsoft/Phi-4-mini-instruct
Validity Rate: 1.0
Invalid Number: 0/6608

utter-project/EuroLLM-9B-Instruct
Validity Rate: 1.0
Invalid Number: 0/6608

google/gemma-3-27b-it
Validity Rate: 1.0
Invalid Number: 0/6608

Qwen/Qwen2.5-32B-Instruct
Validity Rate: 1.0
Invalid Number: 0/6608

Qwen/Qwen3-32B
Validity Rate: 1.0
Invalid Number: 0/6608

generation cot
gemini-2.0-flash-lite
Validity Rate: 1.0
Invalid Number: 0/6608

gemini-2.0-flash
Validity Rate: 1.0
Invalid Number: 0/6608

CohereLabs/aya-expanse-32b
Validity Rate: 1.0
Invalid Number: 0/6608

microsoft/Phi-4-mini-instruct
Validity Rate: 1.0
Invalid Number: 0/6608

utter-project/EuroLLM-9B-Instruct
Validity Rate: 1.0
Invalid Number: 0/6608

google/gemma-3-27b-it
Validity Rate: 1.0
Invalid Number: 0/6608

Qwen/Qwen2.5-32B-Ins

In [15]:
print(pd.DataFrame(validity_df).to_latex())

\begin{tabular}{llr}
\toprule
 & 0 & 1 \\
\midrule
0 & gemini-2.0-flash-lite & 0.999962 \\
1 & gemini-2.0-flash & 0.999465 \\
2 & CohereLabs/aya-expanse-32b & 0.996028 \\
3 & microsoft/Phi-4-mini-instruct & 0.999580 \\
4 & utter-project/EuroLLM-9B-Instruct & 0.965471 \\
5 & google/gemma-3-27b-it & 0.970666 \\
6 & Qwen/Qwen2.5-32B-Instruct & 0.999733 \\
7 & Qwen/Qwen3-32B & 0.918987 \\
\bottomrule
\end{tabular}



## No Think vs Think

In [16]:
# think vs no think
think_result_dicts = {}
think_result_dicts["direct"] = get_result_dict(['Qwen/Qwen3-32B'],"direct", RESULT_DIR, output_files=True)
think_result_dicts["cot"] = get_result_dict(['Qwen/Qwen3-32B'],"cot", RESULT_DIR, output_files=True)
think_result_dicts["think"] = get_result_dict(['Qwen/Qwen3-32B'],"think", RESULT_DIR, output_files=True)

### Absolute DFs

In [17]:
think_prompting_method_dfs = {}
for prompting_method in ["direct","cot","think"]:
    think_prompting_method_dfs[prompting_method] = []
    summary_df = get_template_summary_df(think_result_dicts[prompting_method])
    think_prompting_method_dfs[prompting_method].append(summary_df.copy())
    display(summary_df.style.map(highlight_gradient))

    # average over language
    summary_df['Prefix']=summary_df.index.str.split('-',n=1).str[0]
    lang_agg_summary_df = summary_df.groupby('Prefix').mean()
    think_prompting_method_dfs[prompting_method].append(lang_agg_summary_df.copy())
    display(lang_agg_summary_df.style.map(highlight_gradient))


    # Calculate Judgement Power
    df_models = lang_agg_summary_df.columns.get_level_values(0).unique()
    gen_judge_power_df = lang_agg_summary_df.copy()
    for df_model in df_models:
        gen_judge_power_df[(df_model,'Jud')] = gen_judge_power_df[[(df_model,  'JY'),(df_model,  'JN')]].apply(lambda x: hmean(x),axis=1)
    ordered_cols = [(df_model, col) for df_model in df_models for col in ['Gen', 'JY', 'JN', 'Jud']]
    gen_judge_power_df = gen_judge_power_df[ordered_cols]

    # Calculate Model Power
    df_models = gen_judge_power_df.columns.get_level_values(0).unique()
    model_power_df = gen_judge_power_df.copy()
    for df_model in df_models:
        model_power_df[(df_model,'MP')] = model_power_df[[(df_model,  'Gen'),(df_model,  'Jud')]].apply(lambda x: hmean(x),axis=1)
    ordered_cols = [(df_model, col) for df_model in df_models for col in ['Gen', 'JY','JN', 'Jud','MP']]
    model_power_df = model_power_df[ordered_cols]
    think_prompting_method_dfs[prompting_method].append(model_power_df.copy())
    display(model_power_df.style.map(highlight_gradient))


    # Calculate Model Power Across langauges
    df_models = model_power_df.columns.get_level_values(0).unique()
    mean_model_power_df = model_power_df.copy()

    # We Remove English
    mean_model_power_df = mean_model_power_df[mean_model_power_df.index!='eng']
    mean_model_power_df.loc['hmean'] = mean_model_power_df.apply(lambda x: hmean(x),axis=0)
    mean_model_power_df.loc['mean'] = mean_model_power_df.mean()

    ordered_cols = [(df_model, col) for df_model in df_models for col in ['Gen','JY','JN','Jud','MP']]
    mean_model_power_df = mean_model_power_df[ordered_cols]
    mean_model_power_df = mean_model_power_df.loc[['hmean','mean']]
    think_prompting_method_dfs[prompting_method].append(mean_model_power_df.copy())
    display(mean_model_power_df.style.map(highlight_gradient))

    print()
    print()
    print()


/var/folders/b1/j3sqn5ms737fg97dhk9w3b5m0000gn/T/ipykernel_67492/3206989047.py:18: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  jn = hmean([x['accuracy'] for x in results['judge'] if x['bucket_key'].endswith('_No')])


/var/folders/b1/j3sqn5ms737fg97dhk9w3b5m0000gn/T/ipykernel_67492/3206989047.py:18: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  jn = hmean([x['accuracy'] for x in results['judge'] if x['bucket_key'].endswith('_No')])


/var/folders/b1/j3sqn5ms737fg97dhk9w3b5m0000gn/T/ipykernel_67492/3206989047.py:18: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  jn = hmean([x['accuracy'] for x in results['judge'] if x['bucket_key'].endswith('_No')])


### Max/Min DFs

In [18]:
# CoT vs Think Max/Min
df1 = think_prompting_method_dfs['think'][0]
df2 = think_prompting_method_dfs['cot'][0]
assert df1.index.equals(df2.index)

columns = pd.MultiIndex.from_tuples([
    (col[0].removesuffix('_cot'), col[1]) for col in df1.columns
])
df_diff = pd.DataFrame(df1.values - df2.values, index=df1.index, columns=columns)
display(df_diff.style.map(highlight_accuracy_pos_neg))

# average over language
df_diff['Prefix']=df_diff.index.str.split('-',n=1).str[0]
max_lang_agg_summary_df = df_diff.groupby('Prefix').max()
display(max_lang_agg_summary_df.style.map(highlight_accuracy_pos_neg))
print(max_lang_agg_summary_df.to_latex())

min_lang_agg_summary_df = df_diff.groupby('Prefix').min()
display(min_lang_agg_summary_df.style.map(highlight_accuracy_pos_neg))
print(min_lang_agg_summary_df.to_latex())


\begin{tabular}{lrrr}
\toprule
 & \multicolumn{3}{r}{Qwen/Qwen3-32B_think} \\
 & Gen & JY & JN \\
Prefix &  &  &  \\
\midrule
ara & 0.147000 & 0.026000 & 0.228000 \\
eng & 0.000000 & 0.000000 & 0.000000 \\
fin & 0.200000 & 0.133000 & 0.089000 \\
heb & 0.142000 & -0.021000 & 0.143000 \\
rus & 0.062000 & 0.042000 & 0.095000 \\
tur & 0.227000 & 0.014000 & 0.411000 \\
\bottomrule
\end{tabular}



\begin{tabular}{lrrr}
\toprule
 & \multicolumn{3}{r}{Qwen/Qwen3-32B_think} \\
 & Gen & JY & JN \\
Prefix &  &  &  \\
\midrule
ara & -0.006000 & -0.149000 & -0.169000 \\
eng & 0.000000 & -0.033000 & 0.000000 \\
fin & -0.123000 & -0.061000 & -0.009000 \\
heb & 0.002000 & -0.049000 & -0.162000 \\
rus & -0.198000 & -0.053000 & -0.106000 \\
tur & -0.118000 & -0.118000 & 0.037000 \\
\bottomrule
\end{tabular}



In [19]:
# Direct vs Think Max/Min
df1 = think_prompting_method_dfs['think'][0]
df2 = think_prompting_method_dfs['direct'][0]
assert df1.index.equals(df2.index)

columns = pd.MultiIndex.from_tuples([
    (col[0].removesuffix('_direct'), col[1]) for col in df1.columns
])
df_diff = pd.DataFrame(df1.values - df2.values, index=df1.index, columns=columns)
display(df_diff.style.map(highlight_accuracy_pos_neg))

# average over language
df_diff['Prefix']=df_diff.index.str.split('-',n=1).str[0]
max_lang_agg_summary_df = df_diff.groupby('Prefix').max()
display(max_lang_agg_summary_df.style.map(highlight_accuracy_pos_neg))
print(max_lang_agg_summary_df.to_latex())

min_lang_agg_summary_df = df_diff.groupby('Prefix').min()
display(min_lang_agg_summary_df.style.map(highlight_accuracy_pos_neg))
print(min_lang_agg_summary_df.to_latex())


\begin{tabular}{lrrr}
\toprule
 & \multicolumn{3}{r}{Qwen/Qwen3-32B_think} \\
 & Gen & JY & JN \\
Prefix &  &  &  \\
\midrule
ara & 0.619000 & 0.252000 & 0.416000 \\
eng & 0.020000 & 0.434000 & 0.000000 \\
fin & 0.276000 & 0.448000 & 0.477000 \\
heb & 0.352000 & 0.281000 & 0.195000 \\
rus & 0.322000 & 0.379000 & 0.125000 \\
tur & 0.285000 & 0.198000 & 0.529000 \\
\bottomrule
\end{tabular}



\begin{tabular}{lrrr}
\toprule
 & \multicolumn{3}{r}{Qwen/Qwen3-32B_think} \\
 & Gen & JY & JN \\
Prefix &  &  &  \\
\midrule
ara & -0.385000 & -0.143000 & 0.046000 \\
eng & 0.000000 & 0.011000 & 0.000000 \\
fin & -0.289000 & -0.034000 & -0.938000 \\
heb & 0.008000 & -0.053000 & -0.115000 \\
rus & 0.013000 & 0.085000 & -0.111000 \\
tur & -0.212000 & -0.118000 & -0.055000 \\
\bottomrule
\end{tabular}



### Difference DFs

In [20]:
# Think vs CoT
assert len(think_prompting_method_dfs['think']) == len(think_prompting_method_dfs['cot'])
num = len(think_prompting_method_dfs['think'])
for i in range(num):
    df1 = think_prompting_method_dfs['think'][i]
    df2 = think_prompting_method_dfs['cot'][i]
    assert df1.index.equals(df2.index)

    columns = pd.MultiIndex.from_tuples([
        (col[0].removesuffix('_think'), col[1]) for col in df1.columns
    ])
    df_diff = pd.DataFrame(df1.values - df2.values, index=df1.index, columns=columns)
    display(df_diff.style.map(highlight_accuracy_pos_neg))
    print(df_diff.round(3).to_latex())

\begin{tabular}{lrrr}
\toprule
 & \multicolumn{3}{r}{Qwen/Qwen3-32B} \\
 & Gen & JY & JN \\
category &  &  &  \\
\midrule
eng-com-1 & 0.000000 & -0.005000 & 0.000000 \\
eng-com-2 & 0.000000 & 0.000000 & NaN \\
eng-com-3 & 0.000000 & -0.033000 & NaN \\
ara-com-1 & -0.006000 & -0.084000 & 0.200000 \\
ara-com-2 & 0.063000 & -0.149000 & 0.228000 \\
ara-com-3 & 0.147000 & 0.026000 & 0.078000 \\
ara-number_gender_plurality_agreement & 0.026000 & -0.133000 & -0.169000 \\
heb-number_gender_plurality_agreement & 0.142000 & -0.049000 & -0.162000 \\
heb-com-3 & 0.002000 & -0.046000 & 0.025000 \\
heb-com-2 & 0.038000 & -0.049000 & 0.143000 \\
heb-com-1 & 0.037000 & -0.021000 & -0.028000 \\
rus-com-1 & -0.015000 & -0.005000 & 0.017000 \\
rus-com-2 & 0.062000 & -0.036000 & 0.095000 \\
rus-com-3 & -0.198000 & -0.053000 & 0.049000 \\
rus-motion_verbs & -0.044000 & 0.042000 & -0.106000 \\
tur-verb_plurality_agreement_evidentiality & 0.227000 & 0.014000 & 0.119000 \\
tur-com-1 & 0.158000 & -0.078000 & 0

\begin{tabular}{lrrr}
\toprule
 & \multicolumn{3}{r}{Qwen/Qwen3-32B} \\
 & Gen & JY & JN \\
Prefix &  &  &  \\
\midrule
ara & 0.057000 & -0.085000 & 0.084000 \\
eng & 0.000000 & -0.013000 & 0.000000 \\
fin & -0.003000 & 0.005000 & 0.034000 \\
heb & 0.055000 & -0.041000 & -0.006000 \\
rus & -0.049000 & -0.013000 & 0.014000 \\
tur & 0.088000 & -0.042000 & 0.155000 \\
\bottomrule
\end{tabular}



\begin{tabular}{lrrrrr}
\toprule
 & \multicolumn{5}{r}{Qwen/Qwen3-32B} \\
 & Gen & JY & JN & Jud & MP \\
Prefix &  &  &  &  &  \\
\midrule
ara & 0.057000 & -0.085000 & 0.084000 & -0.009000 & 0.042000 \\
eng & 0.000000 & -0.013000 & 0.000000 & -0.006000 & -0.003000 \\
fin & -0.003000 & 0.005000 & 0.034000 & 0.023000 & 0.003000 \\
heb & 0.055000 & -0.041000 & -0.006000 & -0.022000 & 0.027000 \\
rus & -0.049000 & -0.013000 & 0.014000 & 0.001000 & -0.029000 \\
tur & 0.088000 & -0.042000 & 0.155000 & 0.078000 & 0.086000 \\
\bottomrule
\end{tabular}



\begin{tabular}{lrrrrr}
\toprule
 & \multicolumn{5}{r}{Qwen/Qwen3-32B} \\
 & Gen & JY & JN & Jud & MP \\
Prefix &  &  &  &  &  \\
\midrule
hmean & 0.030000 & -0.041000 & 0.057000 & 0.013000 & 0.026000 \\
mean & 0.030000 & -0.036000 & 0.056000 & 0.014000 & 0.026000 \\
\bottomrule
\end{tabular}



In [21]:
# Direct vs Think
assert len(think_prompting_method_dfs['think']) == len(think_prompting_method_dfs['direct'])
num = len(think_prompting_method_dfs['think'])
for i in range(num):
    df1 = think_prompting_method_dfs['think'][i]
    df2 = think_prompting_method_dfs['direct'][i]
    assert df1.index.equals(df2.index)

    columns = pd.MultiIndex.from_tuples([
        (col[0].removesuffix('_think'), col[1]) for col in df1.columns
    ])
    df_diff = pd.DataFrame(df1.values - df2.values, index=df1.index, columns=columns)
    display(df_diff.style.map(highlight_accuracy_pos_neg))
    print(df_diff.round(3).to_latex())

\begin{tabular}{lrrr}
\toprule
 & \multicolumn{3}{r}{Qwen/Qwen3-32B} \\
 & Gen & JY & JN \\
category &  &  &  \\
\midrule
eng-com-1 & 0.020000 & 0.011000 & 0.000000 \\
eng-com-2 & 0.000000 & 0.095000 & NaN \\
eng-com-3 & 0.000000 & 0.434000 & NaN \\
ara-com-1 & 0.619000 & 0.252000 & 0.164000 \\
ara-com-2 & 0.366000 & -0.016000 & 0.416000 \\
ara-com-3 & -0.385000 & -0.143000 & 0.046000 \\
ara-number_gender_plurality_agreement & -0.064000 & -0.052000 & 0.149000 \\
heb-number_gender_plurality_agreement & 0.088000 & -0.053000 & 0.195000 \\
heb-com-3 & 0.120000 & 0.144000 & 0.023000 \\
heb-com-2 & 0.008000 & 0.281000 & -0.018000 \\
heb-com-1 & 0.352000 & 0.203000 & -0.115000 \\
rus-com-1 & 0.013000 & 0.100000 & 0.000000 \\
rus-com-2 & 0.097000 & 0.086000 & 0.125000 \\
rus-com-3 & 0.322000 & 0.379000 & 0.015000 \\
rus-motion_verbs & 0.173000 & 0.085000 & -0.111000 \\
tur-verb_plurality_agreement_evidentiality & 0.285000 & 0.189000 & 0.017000 \\
tur-com-1 & -0.007000 & -0.027000 & 0.418000 \\

\begin{tabular}{lrrr}
\toprule
 & \multicolumn{3}{r}{Qwen/Qwen3-32B} \\
 & Gen & JY & JN \\
Prefix &  &  &  \\
\midrule
ara & 0.134000 & 0.010000 & 0.194000 \\
eng & 0.007000 & 0.180000 & 0.000000 \\
fin & 0.009000 & 0.114000 & -0.105000 \\
heb & 0.142000 & 0.144000 & 0.021000 \\
rus & 0.151000 & 0.163000 & 0.007000 \\
tur & -0.020000 & 0.079000 & 0.184000 \\
\bottomrule
\end{tabular}



\begin{tabular}{lrrrrr}
\toprule
 & \multicolumn{5}{r}{Qwen/Qwen3-32B} \\
 & Gen & JY & JN & Jud & MP \\
Prefix &  &  &  &  &  \\
\midrule
ara & 0.134000 & 0.010000 & 0.194000 & 0.094000 & 0.132000 \\
eng & 0.007000 & 0.180000 & 0.000000 & 0.101000 & 0.057000 \\
fin & 0.009000 & 0.114000 & -0.105000 & -0.006000 & 0.007000 \\
heb & 0.142000 & 0.144000 & 0.021000 & 0.083000 & 0.123000 \\
rus & 0.151000 & 0.163000 & 0.007000 & 0.088000 & 0.133000 \\
tur & -0.020000 & 0.079000 & 0.184000 & 0.141000 & 0.047000 \\
\bottomrule
\end{tabular}



\begin{tabular}{lrrrrr}
\toprule
 & \multicolumn{5}{r}{Qwen/Qwen3-32B} \\
 & Gen & JY & JN & Jud & MP \\
Prefix &  &  &  &  &  \\
\midrule
hmean & 0.086000 & 0.095000 & 0.065000 & 0.080000 & 0.088000 \\
mean & 0.084000 & 0.101000 & 0.061000 & 0.080000 & 0.088000 \\
\bottomrule
\end{tabular}



### Validity Rate

In [22]:
validity_df = []
for scenario in ["generation","judge"]:
    for prompting_method in ["cot","think"]:
        print(scenario, prompting_method)
        print("="*20)
        for model in ['Qwen/Qwen3-32B']:
            num, den = 0, 0
            for summary_file in RESULT_DIR.glob(f"**/{scenario}/**/{model}/{prompting_method}**/**/summary.csv"):
                df = pd.read_csv(summary_file)
                all_row = df[df.bucket_key=="ALL"].squeeze()
                num+=all_row['n_valid']
                den+=all_row['n_total']
            print(model)
            print(f'Validity Rate: {num/den}')
            print(f'Invalid Number: {den-num}/{den}')
            if scenario == "judge" and prompting_method=="cot":
                validity_df.append([model,num/den])
            print()

generation cot
Qwen/Qwen3-32B
Validity Rate: 1.0
Invalid Number: 0/6608

generation think
Qwen/Qwen3-32B
Validity Rate: 1.0
Invalid Number: 0/6608

judge cot
Qwen/Qwen3-32B
Validity Rate: 0.9189870516786983
Invalid Number: 2121/26181

judge think
Qwen/Qwen3-32B
Validity Rate: 0.9998090218097093
Invalid Number: 5/26181



# Radar Plot

In [23]:
# Radar Plot CSV files
model_names = lang_agg_summary_df.columns.get_level_values(0).unique()
num = len(model_names)
val2add = 360/num
angles = [0]
for _ in range(1,num):
    angles.append(angles[-1]+val2add)

for lang in lang_agg_summary_df.index:
    radar_csv = []
    for model_name,angle in zip(model_names,angles):
        gen_val = lang_agg_summary_df.loc[lang,(model_name,'Gen')]
        jy_val = lang_agg_summary_df.loc[lang,(model_name,'JY')]
        jn_val = lang_agg_summary_df.loc[lang,(model_name,'JN')]
        radar_csv.append([lang,angle,model_name,gen_val,jy_val,jn_val])

        radar_df = pd.DataFrame(radar_csv,columns=['lang','angle','model_name','Gen','JY','JN'])
        radar_df.to_csv(f'{lang}.csv',index=False)

In [24]:
# spider plot for each langauge
import plotly.graph_objects as go

lang_agg_summary_df = prompting_method_dfs['cot'][1]
for lang in ['eng', 'ara','heb','rus','tur','fin']:
  fig = go.Figure()
  for scenario in ['Gen','JY','JN']:
    single_scenario_agg_summary_df = lang_agg_summary_df.loc[:,[x for x in lang_agg_summary_df.columns if scenario in x]]
    single_scenario_lang_agg_summary_df = single_scenario_agg_summary_df.loc[lang]
    categories = [x[0] for x in single_scenario_lang_agg_summary_df.index]
    values = single_scenario_lang_agg_summary_df.values
    fig.add_trace(go.Scatterpolar(
          r=values,
          theta=categories,
          fill='toself',
          name=lang+'_'+scenario
    ))
    fig.update_layout(
      polar=dict(
        radialaxis=dict(
          visible=True,
          range=[0, 1]
        )),
        title=lang,
      showlegend=True
    )

  fig.show()

In [25]:
# spider plot for each langauge
import plotly.graph_objects as go

model_power_agg_summary_df = prompting_method_dfs['cot'][2]
for lang in ['eng', 'ara','heb','rus','tur','fin']:
  fig = go.Figure()
  for scenario in ['Gen','Jud']:
    single_scenario_agg_summary_df = model_power_agg_summary_df.loc[:,[x for x in model_power_agg_summary_df.columns if scenario in x]]
    single_scenario_lang_agg_summary_df = single_scenario_agg_summary_df.loc[lang]
    categories = [x[0] for x in single_scenario_lang_agg_summary_df.index]
    values = single_scenario_lang_agg_summary_df.values
    fig.add_trace(go.Scatterpolar(
          r=values,
          theta=categories,
          fill='toself',
          name=lang+'_'+scenario
    ))
    fig.update_layout(
      polar=dict(
        radialaxis=dict(
          visible=True,
          range=[0, 1]
        )),
        title=lang,
      showlegend=True
    )

  fig.show()

# Checklist Table

In [26]:
import pandas as pd
from pathlib import Path
from src.utils.helpers import analyze_result_file

THRESHOLD = 0.6
result_dict = {}
for model_name in MODELS:
    prompting_method = "cot"
    result_dict[model_name] = {}
    result_dict[model_name][prompting_method] = {}

    for lang in ["eng","ara","heb","rus","tur","fin"]:
        lang_dir = RESULT_DIR / lang
        result_dict[model_name][prompting_method][lang] = {}

        for template_dir in lang_dir.glob("*"):
            template_name = template_dir.name
            result_dict[model_name][prompting_method][lang][template_name]={}
            for scenario_dir in template_dir.glob("*"):
                scenario =scenario_dir.name
                result_dict[model_name][prompting_method][lang][template_name][scenario]=[]
                data_gen_ts_dirs = list(scenario_dir.glob("*"))
                assert len(data_gen_ts_dirs)==1
                data_gen_ts_dir = data_gen_ts_dirs[0]
                inference_ts_dirs = list(data_gen_ts_dir.glob(f'{model_name}/{prompting_method}*/*'))
                inference_ts_dirs = [x for x in inference_ts_dirs if x.is_dir()]
                assert len(inference_ts_dirs)==1, f"The number of directories is {len(inference_ts_dirs)} when it should be one. Check {data_gen_ts_dir} with {model_name}/{prompting_method}"
                if len(inference_ts_dirs)!=1:
                    print(f"The number of directories is {len(inference_ts_dirs)} when it should be one. Check {data_gen_ts_dir} with {model_name}/{prompting_method}")
                    print(f"The command to run is:\npython scripts/run_inference.py  --file {data_gen_ts_dir}/sampled_data.json --scenario {'generation' if 'generation' in str(data_gen_ts_dir) else 'judge'} system_method={prompting_method} model={model_name}")
                    continue
                inference_ts_dir = inference_ts_dirs[0]
                result_file = inference_ts_dir/"results.json"
                df_all, summary_df, mismatch_df = analyze_result_file(Path(result_file))
                summary_df_unique = summary_df[~summary_df.bucket_key.str.contains("ALL")]
                summary_df_unique_filtered = summary_df_unique[summary_df_unique["accuracy"]<THRESHOLD]
                summary_df_unique_filtered = summary_df_unique_filtered[['bucket_key','accuracy']]
                if not summary_df_unique_filtered.empty:
                    result_dict[model_name][prompting_method][lang][template_name][scenario] = summary_df_unique_filtered.to_dict(orient='records')


In [27]:
cot_result_dict = get_result_dict(MODELS, "cot", RESULT_DIR, threshold = 0.6)
        
# Flatten and summarize
records = []

for model_name, prompts in cot_result_dict.items():
    for prompt_type, langs in prompts.items():
        for lang, categories in langs.items():
            for category, results in categories.items():
                gen = 'X' if results.get('generation') else ''
                judge_keys = [b['bucket_key'] for b in results.get('judge', [])]
                jy = 'X' if any(k.endswith('_Yes') for k in judge_keys) else ''
                jn = 'X' if any(k.endswith('_No') for k in judge_keys) else ''
                for measure, val in zip(['Gen', 'JY', 'JN'], [gen, jy, jn]):
                    records.append({
                        'category': lang+'-'+convert_template_name(category),
                        'model': model_name+"_"+prompt_type,
                        'measure': measure,
                        'value': val
                    })

df = pd.DataFrame(records)
summary_df = df.pivot(index='category', columns=['model', 'measure'], values='value')
lang_order = {'eng': 0, 'ara': 1, 'heb': 2, 'rus': 3, 'tur': 4, 'fin': 5}
summary_df_sorted = summary_df.sort_index(key=lambda idx: idx.map(lambda x: lang_order[x.split('-')[0]]))
summary_df_sorted

model                                      gemini-2.0-flash-lite_cot        \
measure                                                          Gen JY JN   
category                                                                     
eng-com-1                                                                    
eng-com-2                                                                    
eng-com-3                                                                    
ara-com-1                                                                X   
ara-com-2                                                          X     X   
ara-com-3                                                          X  X  X   
ara-number_gender_plurality_agreement                                 X  X   
heb-number_gender_plurality_agreement                                        
heb-com-3                                                                    
heb-com-2                                                                X   
heb-com-1                                                                    
rus-com-1                                                                    
rus-com-2                                                                X   
rus-com-3                                                          X     X   
rus-motion_verbs                                                   X     X   
tur-verb_plurality_agreement_evidentiality                               X   
tur-com-1                                                          X     X   
tur-com-2                                                                    
tur-com-3                                                                X   
tur-vowel_harmony                                                        X   
fin-vowel_harmony                                                  X         
fin-com-3                                                          X     X   
fin-com-2                                                                    
fin-com-1                                                                    
fin-lexical_casing_city                                            X     X   

model                                      gemini-2.0-flash_cot        \
measure                                                     Gen JY JN   
category                                                                
eng-com-1                                                               
eng-com-2                                                               
eng-com-3                                                               
ara-com-1                                                           X   
ara-com-2                                                           X   
ara-com-3                                                        X  X   
ara-number_gender_plurality_agreement                         X  X  X   
heb-number_gender_plurality_agreement                               X   
heb-com-3                                                               
heb-com-2                                                        X      
heb-com-1                                                               
rus-com-1                                                               
rus-com-2                                                               
rus-com-3                                                           X   
rus-motion_verbs                                                 X  X   
tur-verb_plurality_agreement_evidentiality                          X   
tur-com-1                                                           X   
tur-com-2                                                               
tur-com-3                                                               
tur-vowel_harmony                                                   X   
fin-vowel_harmony                                                   X   
fin-com-3                                                      

In [28]:
lang_dict = cot_result_dict['gemini-2.0-flash']['cot']

flattenned_bucket_accuracies = []
for lang in lang_dict:
    for template in lang_dict[lang]:
        for scenario in lang_dict[lang][template]:
            bucket_list = lang_dict[lang][template][scenario]
            for bucket_dict in bucket_list:
                flattenned_bucket_accuracies.append({'lang':lang,
                                                     'template':template,
                                                     'scenario':scenario,
                                                     'bucket':bucket_dict['bucket_key'],
                                                     'accuracy':bucket_dict['accuracy']})

flattenned_bucket_accuracies_df = pd.DataFrame(flattenned_bucket_accuracies)
flattenned_bucket_accuracies_df = flattenned_bucket_accuracies_df.sort_values(['accuracy','lang','scenario'],ascending=True)
flattenned_bucket_accuracies_df

,lang,template,scenario,bucket,accuracy
15,fin,lexical_casing_city,judge,noun_allative_No,0.000000
16,fin,lexical_casing_city,judge,noun_illative_No,0.000000
20,rus,adjective_case_plurality_gender_agreement,judge,noun_FEM_PL_NOM_adjective_DAT_SG_FEM_No,0.000000
18,fin,lexical_casing_city,generation,noun_illative,0.058824
24,rus,motion_verbs,judge,place_ACC_verb_unidirectional_SG_No,0.140000
7,ara,number_gender_plurality_agreement,judge,number_FEM_noun_FEM_PL_No,0.240000
5,ara,verb_gender_plurality_agreement_imperative,judge,noun_MASC_DU_verb_MASC_PL_IMP_No,0.300000
27,tur,verb_plurality_agreement_evidentiality,judge,noun_PL_verb_SG_PST_INFR_No,0.300000
1,ara,adjective_gender_plurality_agreement,judge,noun_FEM_DU_NHUM_adjective_MASC_DU_NOM_No,0.320000
10,ara,verb_gender_plurality_agreement_indicative,judge,noun_FEM_DU_verb_MASC_DU_PFV_IND_No,0.360000


In [ ]:
# get file of failing evaluation units
flattenned_bucket_accuracies_df.to_csv('GF2-failing-evaulation-units.csv',index=False)

In [ ]:
# compute pcts of languages of failing evaluation units
flattenned_bucket_accuracies_df.lang.value_counts(normalize=True)

lang
ara    0.413793
fin    0.206897
rus    0.172414
tur    0.137931
heb    0.068966
Name: proportion, dtype: float64

In [ ]:
# compute pcts of scenarios of failing evaluation units
flattenned_bucket_accuracies_df.bucket.apply(lambda x: 'JN' if x.endswith('No') else 'JY' if x.endswith('Yes') else 'Gen').value_counts(normalize=True)

bucket
JN     0.724138
Gen    0.137931
JY     0.137931
Name: proportion, dtype: float64

In [ ]:
# Get the most common failing evaluation units
buckets_models = {}

for model_name, prompts in cot_result_dict.items():
    for prompt_type, langs in prompts.items():
        for lang, categories in langs.items():
            for template_name, results in categories.items():
                gen_keys = [b['bucket_key'] for b in results.get('generation', [])]
                judge_keys = [b['bucket_key'] for b in results.get('judge', [])]
                all_keys = gen_keys + judge_keys
                for val in all_keys:
                    category = lang+'##'+template_name+'##'+val
                    model_prompt = model_name+"_"+prompt_type
                    buckets_models[category] = buckets_models.get(category,[]) + [model_prompt]

buckets_counts = {k:len(v) for (k,v) in buckets_models.items()}
sorted(buckets_counts.items(),key=lambda x: -x[1])

[('ara##adjective_gender_plurality_agreement##noun_FEM_PL_NHUM_adjective_FEM_PL_NOM_No',
  8),
 ('ara##verb_gender_plurality_agreement_imperative##noun_MASC_DU_verb_MASC_PL_IMP_No',
  8),
 ('ara##verb_gender_plurality_agreement_indicative##noun_MASC_DU_verb_MASC_PL_PFV_IND_No',
  8),
 ('fin##lexical_casing_city##noun_allative', 8),
 ('fin##lexical_casing_city##noun_allative_No', 8),
 ('fin##lexical_casing_city##noun_illative_No', 8),
 ('rus##motion_verbs##place_ACC_verb_unidirectional_SG_No', 8),
 ('tur##vowel_harmony##place_LOC_SG_No', 8),
 ('tur##verb_plurality_agreement_evidentiality##noun_PL_verb_SG_PST_INFR_No',
  8),
 ('ara##verb_gender_plurality_agreement_imperative##noun_FEM_DU_verb_FEM_DU_IMP',
  7),
 ('ara##verb_gender_plurality_agreement_imperative##noun_MASC_DU_verb_MASC_DU_IMP',
  7),
 ('ara##verb_gender_plurality_agreement_imperative##noun_FEM_DU_verb_FEM_PL_IMP_No',
  7),
 ('ara##number_gender_plurality_agreement##number_FEM_noun_FEM_PL_No', 7),
 ('ara##verb_gender_plura